# 03 — Test market-data (pub/sub channel `ticks`)

Smoke test du container `fxvol-market-data` — étape 3/5. Valide que **l'engine publie bien sur le channel Redis `ticks`** avec le format JSON attendu, dans la cadence attendue (throttle 200ms par symbol).

**Pourquoi c'est critique** : le channel `ticks` est ce que `api/ws/redis_bridge.py` consume pour pousser les ticks au frontend via WebSocket `/ws/ticks`. Le notebook 02 a validé le **cache** Redis (`SET` keys), ce notebook valide le **stream** (`PUBLISH` sur channel). Sans ce stream, le `ChartPanel` du dashboard reste figé même si les keys cache se rafraîchissent.

**Couvre** :
1. Subscribe au channel `ticks` → réception ≥ 5 messages en 5s
2. Chaque message est un JSON valide avec les 5 clés attendues : `symbol`, `bid`, `ask`, `mid`, `timestamp`
3. Types corrects : `symbol=str`, `bid/ask/mid=float`, `timestamp=str` parsable ISO-8601
4. Cohérence par message : `bid ≤ mid ≤ ask`
5. Throttle 200ms : intervalle médian entre 2 messages consécutifs ∈ [150ms, 500ms]. Trop court = throttle non respecté ; trop long = ticks IB pas assez fréquents (peut être normal en off-hours)

**Préreq** :
- Notebook 01 + 02 verts
- Marché ouvert recommandé (off-hours = ticks IB rares = on peut ne pas atteindre 5 messages en 5s ; le test reportera FAIL légitimement, à re-tester en heures de marché)
- Redis exposé sur `localhost:6380`

**Référence** : `src/bus/publisher.py:publish_tick` (throttle 200ms, payload format), `src/bus/channels.py:CH_TICKS = "ticks"`, `src/engines/market_data/engine.py` (boucle 100ms qui appelle publish_tick).

## Setup — connexion + subscriber

In [1]:
import json
import time
from datetime import datetime
from statistics import median

import redis

REDIS_URL = "redis://localhost:6380/0"
CHANNEL = "ticks"
EXPECTED_KEYS = {"symbol", "bid", "ask", "mid", "timestamp"}

# Throttle attendu : 200ms par symbole. On accepte un peu de latence
# réseau / scheduling Python : marge [150, 500] ms.
MIN_INTERVAL_MS = 150
MAX_INTERVAL_MS = 500

# Cible : ≥ 5 messages en 5s. À 200ms throttle = ~25 messages max théorique.
MIN_MESSAGES = 5
LISTEN_DURATION_S = 5.0

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps' = redis healthy")
print(f"Connected -> {REDIS_URL}, channel = {CHANNEL!r}")

Connected -> redis://localhost:6380/0, channel = 'ticks'


## 1. Subscribe + listen pendant 5s

**Ce que tu dois voir** : ≥ 5 messages reçus en 5 secondes. À 200ms de throttle théorique, on devrait avoir ~25 messages max. 5 = plancher confortable.

**Si FAIL (0 ou < 5 messages)** :
- 0 message → engine ne publie pas (cf. logs `docker logs fxvol-market-data`) OU on est connecté à la mauvaise DB Redis
- 1-4 messages → marché peu actif (off-hours) ou throttle plus aggressif que prévu
- Re-tester en heures de marché avant de conclure à un bug

In [2]:
print(f"== 1. subscribe + listen {LISTEN_DURATION_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe(CHANNEL)

messages = []  # liste de (timestamp_recv_ms, raw_data)
deadline = time.perf_counter() + LISTEN_DURATION_S
while time.perf_counter() < deadline:
    msg = sub.get_message(timeout=0.1)
    if msg and msg.get("type") == "message":
        messages.append((time.perf_counter() * 1000, msg["data"]))

sub.unsubscribe(CHANNEL)
sub.close()

record(f"≥ {MIN_MESSAGES} messages reçus en {LISTEN_DURATION_S}s",
       len(messages) >= MIN_MESSAGES,
       f"{len(messages)} messages")

== 1. subscribe + listen 5.0s ==
  [OK  ] ≥ 5 messages reçus en 5.0s  | 25 messages


## 2. Chaque message est un JSON valide avec les 5 clés attendues

**Schema attendu** (cf. `src/bus/publisher.py:publish_tick`) :
```json
{
  "symbol": "EURUSD",
  "bid": 1.16967,
  "ask": 1.16968,
  "mid": 1.169675,
  "timestamp": "2026-04-28T14:27:43.834417+00:00"
}
```

**Si FAIL** : payload corrompu, engine modifié sans synchroniser le bus, ou on consume le mauvais channel.

In [3]:
print("== 2. JSON parsing + schema ==")

if not messages:
    record("JSON valide + clés attendues", False, "aucun message à inspecter (cf. §1)")
else:
    parsed = []
    parse_errors = 0
    schema_errors = 0
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if set(obj.keys()) >= EXPECTED_KEYS:
            parsed.append(obj)
        else:
            schema_errors += 1
    record("tous les messages JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs de parse / {len(messages)}")
    record(f"tous ont les clés {sorted(EXPECTED_KEYS)}",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(messages)}")
    print(f"  [INFO] sample message[0] = {parsed[0] if parsed else '<no parsed>'}")

== 2. JSON parsing + schema ==
  [OK  ] tous les messages JSON-parseables  | 0 erreurs de parse / 25
  [OK  ] tous ont les clés ['ask', 'bid', 'mid', 'symbol', 'timestamp']  | 0 schémas incomplets / 25
  [INFO] sample message[0] = {'symbol': 'EURUSD', 'bid': 1.16998, 'ask': 1.17, 'mid': 1.1699899999999999, 'timestamp': '2026-04-28T14:31:30.688946Z'}


## 3. Types corrects + timestamp ISO-8601 parsable

**Ce que tu dois voir** : `symbol` est str, `bid/ask/mid` sont des floats numériques finis, `timestamp` est une string parsable par `datetime.fromisoformat`.

In [4]:
print("== 3. types + timestamp ISO ==")

if not messages:
    record("types + ISO", False, "skip (cf. §1)")
else:
    type_errors = 0
    ts_errors = 0
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(obj.get("symbol"), str):
            type_errors += 1
            continue
        if not all(isinstance(obj.get(k), (int, float)) for k in ("bid", "ask", "mid")):
            type_errors += 1
            continue
        ts = obj.get("timestamp")
        if not isinstance(ts, str):
            ts_errors += 1
            continue
        try:
            datetime.fromisoformat(ts.replace("Z", "+00:00"))
        except ValueError:
            ts_errors += 1
    record("types corrects (symbol str, bid/ask/mid float)",
           type_errors == 0,
           f"{type_errors} erreurs / {len(messages)}")
    record("timestamp ISO-8601 parsable",
           ts_errors == 0,
           f"{ts_errors} erreurs / {len(messages)}")

== 3. types + timestamp ISO ==
  [OK  ] types corrects (symbol str, bid/ask/mid float)  | 0 erreurs / 25
  [OK  ] timestamp ISO-8601 parsable  | 0 erreurs / 25


## 4. Cohérence par message — `bid ≤ mid ≤ ask`

Invariant simple sur chaque tick : le mid est entre les deux côtés du book. Si un seul message viole ça, c'est un bug côté engine (calcul de mid fait n'importe comment).

**Tolérance flottante** : on accepte une marge ε de `1e-9` (un milliardième) pour absorber les imprécisions de représentation float32/64 (ex: `mid` calculé comme `(bid+ask)/2` et stocké en float — peut être à 1 ULP du vrai milieu).

In [5]:
print("== 4. cohérence bid ≤ mid ≤ ask par message ==")

EPS = 1e-9

if not messages:
    record("cohérence bid/mid/ask", False, "skip (cf. §1)")
else:
    coh_errors = 0
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        bid, mid, ask = obj.get("bid"), obj.get("mid"), obj.get("ask")
        if not all(isinstance(v, (int, float)) for v in (bid, mid, ask)):
            continue
        if not (bid - EPS <= mid <= ask + EPS):
            coh_errors += 1
    record("bid ≤ mid ≤ ask sur tous les messages",
           coh_errors == 0,
           f"{coh_errors} violations / {len(messages)}")

== 4. cohérence bid ≤ mid ≤ ask par message ==
  [OK  ] bid ≤ mid ≤ ask sur tous les messages  | 0 violations / 25


## 5. Throttle 200ms — intervalle médian dans la fenêtre attendue

Le `publish_tick` (`src/bus/publisher.py`) throttle à `TICK_PUBLISH_THROTTLE_MS = 200` par symbole. Donc 2 messages consécutifs sur `ticks` ne peuvent **pas** être plus rapprochés que 200ms.

**Fenêtre acceptée [150, 500] ms** :
- 150ms = marge inférieure (latence Python/scheduling/network)
- 500ms = marge supérieure (cas où IB pousse moins de 5 ticks/s, donc moins de PUBLISH même sans throttle)

**Métrique** : médiane des intervalles, plus robuste qu'une moyenne (ne se fait pas bouger par un outlier).

**Si la médiane sort de la fenêtre** :
- < 150ms → throttle ne s'applique pas (bug `publish_tick` ou plusieurs engines tournent en parallèle sur le même symbol)
- > 500ms → ticks IB rares (off-hours, marché peu actif). Pas un bug du smoke, juste un marché lent. Acceptable.

In [6]:
print("== 5. throttle ~200ms (médiane des intervalles) ==")

if len(messages) < 2:
    record("throttle", False, "skip (besoin de ≥ 2 messages)")
else:
    timestamps_ms = [t for t, _ in messages]
    intervals_ms = [t2 - t1 for t1, t2 in zip(timestamps_ms, timestamps_ms[1:])]
    med = median(intervals_ms)
    record(f"intervalle médian ∈ [{MIN_INTERVAL_MS}, {MAX_INTERVAL_MS}] ms",
           MIN_INTERVAL_MS <= med <= MAX_INTERVAL_MS,
           f"median = {med:.0f}ms ({len(intervals_ms)} intervals, min={min(intervals_ms):.0f}, max={max(intervals_ms):.0f})")

== 5. throttle ~200ms (médiane des intervalles) ==
  [OK  ] intervalle médian ∈ [150, 500] ms  | median = 205ms (24 intervals, min=202, max=208)


## Récap final

In [7]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — pub/sub stream sain. Pass au notebook 04 (WebSocket bridge api).")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
≥ 5 messages reçus en 5.0s                              OK      25 messages
tous les messages JSON-parseables                       OK      0 erreurs de parse / 25
tous ont les clés ['ask', 'bid', 'mid', 'symbol', 'timestamp'] OK      0 schémas incomplets / 25
types corrects (symbol str, bid/ask/mid float)          OK      0 erreurs / 25
timestamp ISO-8601 parsable                             OK      0 erreurs / 25
bid ≤ mid ≤ ask sur tous les messages                   OK      0 violations / 25
intervalle médian ∈ [150, 500] ms                       OK      median = 205ms (24 intervals, min=202, max=208)
--------------------------------------------------------------------------------------------------------------

7 OK / 0 FAIL  (7 total)

OK — pub/sub stream sain. Pass au notebook 04 (WebSocket bridge ap

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| 0 message en 5s | engine pas démarré, ou Redis URL incorrecte côté market-data | `docker logs fxvol-market-data` ; vérifier `REDIS_URL` dans la config compose |
| < 5 messages | throttle plus strict OU ticks IB peu fréquents (off-hours) | re-tester en heures de marché ; sinon investiguer `TICK_PUBLISH_THROTTLE_MS` dans `src/bus/publisher.py` |
| `parse_errors > 0` | payload non-JSON publié sur le channel (un autre service spam ?) | grepper `redis.publish.*ticks` dans `src/` pour identifier les producers |
| schéma incomplet | `publish_tick` modifié sans aligner le bus / les consumers | revue diff `src/bus/publisher.py` |
| `type_errors > 0` | corruption ou serializer custom mal configuré | inspecter `_dumps`/`_iso_utc` dans `src/bus/publisher.py` |
| `timestamp` non-ISO | format non standard (Unix ts ? local time ?) | aligner sur ISO-8601 UTC, cf. `_iso_utc` |
| `bid > mid` ou `mid > ask` (cohérence FAIL) | bug calcul mid côté engine | investiguer `src/engines/market_data/engine.py` (`ticker.midpoint()` ou autre) |
| médiane < 150ms | throttle 200ms non respecté | bug `publish_tick` ou doublons (plusieurs market-data instances ?) |
| médiane > 500ms | ticks IB rares (marché lent) | marqueur info, pas critique |